In [ ]:
from Myself_DecisionTree import Decision_tree
import numpy as np
import pandas as pd
from collections import Counter

class Random_Forest:
    """
    Реализует алгоритм Случайного Леса (Random Forest) для задач классификации.

    Ансамбль строит заданное количество независимых деревьев решений на основе
    бутстрап-выборок (случайный выбор строк с возвращением). Финальное предсказание
    определяется путем мажоритарного голосования (большинством голосов).

    Аргументы инициализации :
    ----------------------------------
    n_estimators : int, default=100
        Количество деревьев решений в ансамбле (размер леса).
    max_depth : int, default=10
        Максимальная глубина каждого одиночного дерева решений в лесу.
    min_samples_split : int, default=2
        Минимальное количество объектов в узле, необходимое для выполнения сплита
        внутри каждого дерева.
    criterion : str, default='gini'
        Критерий информативности, используемый деревьями для оценки качества разбиения.
        Доступные варианты: 'gini' (индекс Джини), 'entropy' (энтропия Шеннона).

    Основные методы:
    ----------------
    fit(X, y)
        Запускает процесс обучения ансамбля: формирует бутстрап-выборки для каждого
        дерева и обучает self.n_estimators независимых моделей Decision_tree.
    predict(X)
        Предсказывает итоговые метки классов для объектов из переданного датафрейма
        на основе мажоритарного голосования всех деревьев в лесу.

    Внутренние атрибуты:
    --------------------
    self.trees : list
        Список, содержащий все обученные экземпляры класса Decision_tree,
        входящие в ансамбль.
    """

    def __init__(self, n_estimators=100, max_depth=10, min_samples_split=2, criterion='gini'):
        self.n_estimators = n_estimators
        self.max_depth = max_depth
        self.min_samples_split = min_samples_split
        self.criterion = criterion
        self.trees = []

    def fit(self, X, y):
        self.trees = []
        num_samples = len(X)

        for _ in range(self.n_estimators):
            bootstrap_indices = np.random.choice(num_samples, size=num_samples, replace=True)

            X_sample = X.iloc[bootstrap_indices]
            y_sample = y.iloc[bootstrap_indices]

            tree = Decision_tree(
                max_depth=self.max_depth,
                min_samples_split=self.min_samples_split,
                criterion=self.criterion
            )
            tree.fit(X_sample, y_sample)
            self.trees.append(tree)
        return self

    def predict(self, X: pd.DataFrame):
        all_tree_preds = np.array([tree.predict(X) for tree in self.trees])
        final_preds = []
        num_objects = X.shape[0]

        for i in range(num_objects):
            object_preds = all_tree_preds[:, i]
            most_common_pred = Counter(object_preds).most_common(1)[0][0]
            final_preds.append(most_common_pred)

        return np.array(final_preds)


### Сравним результаты работы алгоритма на встроенном наборе данных о винах Wine Dataset

In [ ]:
from sklearn.datasets import load_wine
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier

wine = load_wine()
X = pd.DataFrame(wine.data, columns=wine.feature_names)
y = pd.Series(wine.target)

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.3, random_state = 42)

param = {'max_depth':3, 'min_samples_split':2, 'criterion':'entropy', 'n_estimators':10}

my_rf = Random_Forest(**param).fit(X_train, y_train)
My_predict = my_rf.predict(X_test)

sk_rf = RandomForestClassifier(**param, random_state = 42).fit(X_train, y_train)
SK_predict = sk_rf.predict(X_test)

In [ ]:
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix

print(f"""=== СРАВНЕНИЕ ACCURACY ===
Random_Forest          : {accuracy_score(y_test, My_predict):.4f}
RandomForestClassifier : {accuracy_score(y_test, SK_predict):.4f}

=== ПОДРОБНЫЙ ОТЧЕТ Random_Forest ===
{classification_report(y_test, My_predict, target_names=wine.target_names)}

=== ПОДРОБНЫЙ ОТЧЕТ RandomForestClassifier ===
{classification_report(y_test, SK_predict, target_names=wine.target_names)}

=== МАТРИЦА ОШИБОК Random_Forest ===
{confusion_matrix(y_test, My_predict)}

=== МАТРИЦА ОШИБОК RandomForestClassifier ===
{confusion_matrix(y_test, SK_predict)}""")

=== СРАВНЕНИЕ ACCURACY ===
Random_Forest          : 0.9815
RandomForestClassifier : 0.9074

=== ПОДРОБНЫЙ ОТЧЕТ Random_Forest ===
              precision    recall  f1-score   support

     class_0       0.95      1.00      0.97        19
     class_1       1.00      0.95      0.98        21
     class_2       1.00      1.00      1.00        14

    accuracy                           0.98        54
   macro avg       0.98      0.98      0.98        54
weighted avg       0.98      0.98      0.98        54


=== ПОДРОБНЫЙ ОТЧЕТ RandomForestClassifier ===
              precision    recall  f1-score   support

     class_0       0.86      1.00      0.93        19
     class_1       0.94      0.81      0.87        21
     class_2       0.93      0.93      0.93        14

    accuracy                           0.91        54
   macro avg       0.91      0.91      0.91        54
weighted avg       0.91      0.91      0.91        54


=== МАТРИЦА ОШИБОК Random_Forest ===
[[19  0  0]
 [ 1 20  0

### Результаты сравнительного анализа моделей Случайного Леса (Актуализированные)

#### 1. Метрики эффективности моделей
* **Общая точность (Accuracy):**
  > Разработанный ансамбль **`Random_Forest` продемонстрировал превосходную точность, достигнув Accuracy = 0.9815** (53 верно предсказанных объекта из 54). Он существенно превзошел по метрикам библиотечный аналог `RandomForestClassifier` из `scikit-learn`, который на данном разбиении набрал **Accuracy = 0.9074** (49 верных предсказаний из 54).

* **Качество по классам (на основе Classification Report):**
  * **`class_0`**: Кастомная модель показала безупречную полноту ($Recall = 1.00$) при высокой точности ($Precision = 0.95$). Модель `sklearn` сильнее просела по точности ($Precision = 0.86$), ошибочно отнеся к этому классу 3 чужих объекта.
  * **`class_1`**: Реализованный лес достиг идеальной точности ($Precision = 1.00$) при полноте $Recall = 0.95$. Алгоритм `sklearn` заметно отстал, показав полноту всего $Recall = 0.81$ (пропущено 4 объекта).
  * **`class_2`**: Разработанный ансамбль **безошибочно классифицировал все объекты данного класса**, получив идеальные метрики ($Precision = 1.00$, $Recall = 1.00$, $F_1 = 1.00$). Модель `sklearn` здесь также показала более слабый результат ($F_1 = 0.93$).

---

#### 2. Анализ матриц ошибок (Confusion Matrix)
Матрицы ошибок детально локализуют допущенные моделями просчеты:
* **`Random_Forest` (всего 1 ошибка):**
  * Алгоритм допустил лишь одну ошибку во всей тестовой выборке: ошибочно отправил 1 объект из реального `class_1` в `class_0` (строка 2 в матрице ошибок: `[1 20 0]`). В остальном классификация идеальна.
* **Библиотечный `RandomForestClassifier` (всего 5 ошибок):**
  * Допустил 3 ошибки, отправив объекты из реального `class_1` в `class_0`.
  * Допустил 1 ошибку, отправив объект из реального `class_1` в `class_2`.
  * Допустил 1 ошибку, отправив объект из реального `class_2` в `class_1`.

---

#### 3. Причины превосходства кастомной модели
1. **Полнота перебора признаков:** `Random_Forest` ищет оптимальные разделения по всему пространству признаков без принудительного усечения подмножества столбцов до $\sqrt{D}$. На данном объеме данных это позволило сформировать более строгие и точные правила деления в узлах.
2. **Успешный бэггинг:** Случайные бутстрап-выборки строк для отдельных деревьев сгенерировали оптимальный набор «экспертов». Мажоритарное голосование эффективно сгладило единичные выбросы, оставив итоговую ошибку ансамбля минимальной (1 объект из 54).

#### Итог
Актуализированные тесты подтверждают высокую надежность кастомной архитектуры. Разработанный Случайный лес демонстрирует идеальное качество аппроксимации целевой зависимости на тестовом множестве, значительно опережая базовое коробочное решение из `scikit-learn`.
